# Credit Card Fraud Detection — Exploratory Analysis

This notebook is the **EDA layer** of the project. Modeling, threshold tuning, and deployment live in `src/` and `train.py`.

**Rules enforced by the training pipeline (not this notebook):**
- Stratified 70/15/15 split **before** scaling or resampling
- SMOTE only on training data
- Thresholds tuned only on validation
- Test set scored once at the end


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data_loader import download_dataset, load_raw_dataframe, validate_dataset
from src.eda import run_eda


## Load and validate

The CSV is not stored in Git. `download_dataset()` copies a local file, uses Kaggle credentials, or falls back to the TensorFlow public mirror of the ULB dataset.


In [ ]:
download_dataset()
df = load_raw_dataframe()
report = validate_dataset(df)
report

## Why accuracy is the wrong metric

Fraud is about **0.17%** of rows. A classifier that never predicts fraud is ~99.83% accurate and operationally useless. The training pipeline therefore reports **PR-AUC, recall, precision, and F1**, and it tunes the decision threshold on validation data using a cost model (missed fraud >> false alert).


In [ ]:
n = len(df)
n_fraud = int((df["Class"] == 1).sum())
print(f"rows={n:,}  fraud={n_fraud:,}  fraud_rate={100 * n_fraud / n:.4f}%")
print(f"majority-class accuracy floor={100 * (1 - n_fraud / n):.4f}%")
print("missing values:", int(df.isna().sum().sum()))
print("duplicate rows:", int(df.duplicated().sum()))
df[["Time", "Amount", "Class"]].describe()

## Figures

`run_eda` writes plots to `outputs/figures/` (class imbalance, amount, time, correlations, box plots, fraud patterns). Outliers are **profiled but not removed** — fraud is often an outlier.


In [ ]:
summary = run_eda(df)
summary["fraud_percentage"], summary["n_fraud"], summary["notes"]